# 04 · Visualizer

**Objetivo.** Generar el tablero HTML autocontenido del Visualizer a partir del
documento del Analyzer y verificar que se abre sin conexión ni recursos externos.

**Entrada.** `ANALYZER_OUTPUT` (`analysis/outputs/analyzer_output.json`, contrato
`schema_version` 1.1). Si no existe, se genera con
`analysis.pipeline.run_analysis` a partir de `INPUT_PATHS`: `results.json` si
existe; si no, los que existan de `results-sbom.json` y `results-vuln.json`.

**Salida.** `analysis/outputs/visualizer.html`: un único archivo con el documento
del Analyzer y los assets del tablero (CSS/JS) embebidos.

**Supuestos.**
- El Visualizer no recalcula métricas: solo renderiza el documento validado.
- El HTML es determinista salvo por `meta.generated_at`, que se puede fijar con
  la variable de entorno `GENERATED_AT`.
- Los assets del tablero no referencian recursos externos de terceros.


## 0. Parámetros y rutas

Rutas resueltas desde la raíz del repo (donde está `pyproject.toml`). El
documento de entrada es el que produce el notebook 03; si no existe, se genera
aquí mismo con el mismo pipeline y los mismos `INPUT_PATHS`.


In [ ]:
from pathlib import Path

import os


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return start


def _default_inputs(root: Path) -> list:
    scan = root / "results.json"
    if scan.is_file():
        return [scan]
    candidates = [root / "results-sbom.json", root / "results-vuln.json"]
    return [path for path in candidates if path.is_file()]


REPO_ROOT = _find_repo_root(Path.cwd())
ANALYZER_OUTPUT = REPO_ROOT / "analysis" / "outputs" / "analyzer_output.json"
OUTPUT_HTML = REPO_ROOT / "analysis" / "outputs" / "visualizer.html"
INPUT_PATHS = _default_inputs(REPO_ROOT)

# Reproducibilidad exacta: exporta GENERATED_AT con una cadena ISO-8601.
GENERATED_AT = os.environ.get("GENERATED_AT")

print("REPO_ROOT      :", REPO_ROOT)
print("ANALYZER_OUTPUT:", ANALYZER_OUTPUT)
print("OUTPUT_HTML    :", OUTPUT_HTML)
print("GENERATED_AT   :", GENERATED_AT)


## 1. Preparar el entorno

Se importan el pipeline del Analyzer (para el respaldo) y el Visualizer. El
paquete `miner.visualizer` está instalado, por lo que basta con importarlo.


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.pipeline import run_analysis
from miner.visualizer import build_visualizer, load_document


## 2. Documento del Analyzer

El Visualizer consume el documento del notebook 03. Si no existe (p. ej. al
ejecutar este notebook aislado), se genera con `run_analysis` y
`generated_at=GENERATED_AT` para mantener la reproducibilidad.


In [ ]:
if ANALYZER_OUTPUT.is_file():
    print("Documento existente:", ANALYZER_OUTPUT)
else:
    if not INPUT_PATHS:
        raise FileNotFoundError(
            "No se encontró results.json, results-sbom.json ni results-vuln.json"
        )
    print("No existe; generando con run_analysis a partir de INPUT_PATHS...")
    run_analysis(INPUT_PATHS, ANALYZER_OUTPUT, generated_at=GENERATED_AT)

assert ANALYZER_OUTPUT.is_file(), ANALYZER_OUTPUT
document = load_document(ANALYZER_OUTPUT)

print("schema_version:", document["schema_version"])
print("organización  :", document["meta"]["organization"])
print("origen        :", document["meta"]["source_kind"])


## 3. Generar el tablero HTML

`build_visualizer` embebe el documento y los assets del tablero en un único
HTML. El título por defecto es `Visualizer · <organización>`; aquí se pasa de
forma explícita.


In [ ]:
TITLE = f"Visualizer · {document['meta']['organization']}"
built = build_visualizer(ANALYZER_OUTPUT, OUTPUT_HTML, title=TITLE)

print("Título :", TITLE)
print("HTML   :", built)


## 4. Validación del HTML

Se comprueba que el archivo existe y no está vacío, que incluye el bloque de
datos del Analyzer y que no referencia recursos externos de terceros. Para no
confundir las URLs de los datos con dependencias, el bloque
`<script id="analyzer-data">` se excluye antes de buscar `src="http`,
`href="http` o `@import url(http`.


In [ ]:
html = OUTPUT_HTML.read_text(encoding="utf-8")

assert OUTPUT_HTML.is_file(), OUTPUT_HTML
assert html.strip(), "El HTML generado está vacío"
assert '<script id="analyzer-data" type="application/json">' in html, (
    "Falta el bloque de datos del Analyzer"
)

# Se recorta el contenido JSON embebido: sus URLs de datos son legítimas.
DATA_MARKER = '<script id="analyzer-data" type="application/json">'
data_start = html.index(DATA_MARKER) + len(DATA_MARKER)
data_end = html.index("</script>", data_start)
html_without_data = html[:data_start] + html[data_end:]

EXTERNAL_MARKERS = ('src="http', 'href="http', '@import url(http')
external = [marker for marker in EXTERNAL_MARKERS if marker in html_without_data]
assert not external, f"Recursos externos detectados: {external}"

print("Bloque de datos del Analyzer : OK")
print("Sin recursos externos        : OK")


## 5. Resumen

Datos principales del documento consumido y del HTML generado.


In [ ]:
meta = document["meta"]
datasets = document["datasets"]
risk = datasets["risk_summary"]

try:
    shown = OUTPUT_HTML.relative_to(REPO_ROOT)
except ValueError:
    shown = OUTPUT_HTML

print("Resumen del Visualizer")
print("----------------------")
print("Organización        :", meta["organization"])
print("Repositorios        :", meta["repositories"])
print("Vulnerabilidades    :", len(datasets["vulnerabilities"]))
print("Nota global         :", f"{risk['score']}/10")
print("Datasets consumidos :", len(datasets))
print("HTML                :", shown, f"({OUTPUT_HTML.stat().st_size} bytes)")


## 6. Verificación final

In [ ]:
assert OUTPUT_HTML.is_file(), f"No existe: {OUTPUT_HTML}"
assert OUTPUT_HTML.stat().st_size > 0, f"Archivo vacío: {OUTPUT_HTML}"
print("Visualizer listo:", OUTPUT_HTML)
